# 🔮 04 - AI Forecasting with Databricks ai_forecast

**Predictive Analytics for Ticket Management** - 

## What this notebook does:
- ✅ Prepares time series data from ticket analysis results
- ✅ Uses Databricks `ai_forecast` function for predictive analytics
- ✅ Forecasts ticket volume, priority trends, and system health patterns
- ✅ Creates asset failure prediction models
- ✅ Generates actionable insights for proactive maintenance

**Prerequisites:** Run notebooks 01, 02, and 03 first

## Key Forecasting Use Cases:
- 📈 **Ticket Volume Prediction**: Forecast future ticket volumes by category
- ⚠️ **Critical Issue Forecasting**: Predict when critical issues might spike
- 🔧 **Asset Failure Prediction**: Identify systems likely to fail based on ticket patterns
- 📊 **Resource Planning**: Forecast team workload and resource needs
- 🎯 **Proactive Maintenance**: Schedule maintenance before failures occur

## Databricks ai_forecast Function:
Based on [Microsoft's ai_forecast documentation](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/ai_forecast), this function provides:
- Prophet-like piecewise linear and seasonality modeling
- Automatic frequency detection
- Prediction intervals with confidence levels
- Multi-metric and multi-group forecasting
- Built-in seasonality handling


In [3]:
# Import required libraries
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pandas as pd
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns

# Import configuration
%run ./config

print("✅ Libraries imported and configuration loaded")
print(f"🎯 Using Unity Catalog: {UNITY_CATALOG['catalog_name']}.{UNITY_CATALOG['schema_name']}")
print("🔮 Ready for AI forecasting with Databricks ai_forecast function")


✅ Libraries imported and configuration loaded
🎯 Using Unity Catalog: quickstart_catalog_vkm_external.classify_tickets
🔮 Ready for AI forecasting with Databricks ai_forecast function


In [ ]:
# Load AI analysis results for time series forecasting
print("📊 Step 1: Loading AI analysis results...")

# Load the AI analysis results from previous notebook
df_ai_results = spark.table(TABLES["ai_showcase_results"])

print(f"✅ Loaded {df_ai_results.count()} AI analysis records")
print("\n📋 Available columns:")
df_ai_results.printSchema()

# Display sample data
print("\n📊 Sample AI analysis data:")
display(df_ai_results.limit(3))


📊 Step 1: Loading AI analysis results...


KeyError: 'ai_analysis_results'

In [ ]:
# Prepare time series data for forecasting
print("⏰ Step 2: Preparing time series data...")

# Create daily aggregated data for forecasting
df_daily_tickets = df_ai_results.withColumn(
    "date", date_format(col("ai_showcase_timestamp"), "yyyy-MM-dd")
).groupBy(
    "date",
    "ai_priority_classification",
    "affected_systems"
).agg(
    count("ticket_id").alias("ticket_count"),
    count(when(col("urgency_level") == "High", 1)).alias("high_urgency_count"),
    count(when(col("urgency_level") == "Critical", 1)).alias("critical_urgency_count")
).orderBy("date")

print("✅ Daily aggregated data prepared")
print(f"📅 Date range: {df_daily_tickets.select(min('date'), max('date')).collect()[0]}")

# Display sample aggregated data
print("\n📊 Sample daily aggregated data:")
display(df_daily_tickets.limit(10))


In [ ]:
# Create overall ticket volume time series
print("📈 Step 3: Creating overall ticket volume time series...")

# Aggregate by date for overall volume forecasting
df_volume_ts = df_daily_tickets.groupBy("date").agg(
    sum("ticket_count").alias("total_tickets"),
    sum("high_urgency_count").alias("high_urgency_tickets"),
    sum("critical_urgency_count").alias("critical_urgency_tickets")
).orderBy("date")

# Save to temporary view for ai_forecast function
df_volume_ts.createOrReplaceTempView("ticket_volume_ts")

print("✅ Overall ticket volume time series created")
print(f"📊 Total data points: {df_volume_ts.count()}")

# Display the time series data
print("\n📈 Ticket volume time series:")
display(df_volume_ts)


In [ ]:
# Use ai_forecast to predict future ticket volumes
print("🔮 Step 4: Forecasting future ticket volumes using ai_forecast...")

# Calculate forecast horizon (30 days from last observation)
max_date = df_volume_ts.select(max("date")).collect()[0][0]
forecast_horizon = (datetime.strptime(max_date, "%Y-%m-%d") + timedelta(days=30)).strftime("%Y-%m-%d")

print(f"📅 Forecasting from {max_date} to {forecast_horizon}")

# Use ai_forecast function to predict ticket volumes
forecast_query = f"""
SELECT * FROM AI_FORECAST(
  TABLE(ticket_volume_ts),
  horizon => '{forecast_horizon}',
  time_col => 'date',
  value_col => ARRAY('total_tickets', 'high_urgency_tickets', 'critical_urgency_tickets'),
  prediction_interval_width => 0.95,
  frequency => '1D',
  parameters => '{{"global_floor": 0}}'
)
"""

# Execute the forecast
df_forecast = spark.sql(forecast_query)

print("✅ AI forecast completed")
print(f"🔮 Forecasted {df_forecast.count()} future data points")

# Display forecast results
print("\n🔮 Ticket volume forecast results:")
display(df_forecast.orderBy("date"))


In [ ]:
# Asset failure prediction based on ticket patterns
print("🔧 Step 5: Asset failure prediction using ticket patterns...")

# Create system health indicators from ticket data
df_system_health = df_daily_tickets.groupBy(
    "date", "affected_systems"
).agg(
    sum("ticket_count").alias("total_issues"),
    sum("critical_urgency_count").alias("critical_issues"),
    (sum("critical_urgency_count") / sum("ticket_count")).alias("critical_ratio")
).filter(col("affected_systems") != "Unknown")

# Calculate system health score (lower is better)
df_system_health = df_system_health.withColumn(
    "health_score", 
    col("total_issues") * 0.3 + col("critical_issues") * 0.7 + col("critical_ratio") * 10
).orderBy("affected_systems", "date")

# Save to temporary view
df_system_health.createOrReplaceTempView("system_health_ts")

print("✅ System health time series created")
print(f"🔧 Monitoring {df_system_health.select('affected_systems').distinct().count()} systems")

# Display system health data
print("\n🔧 System health indicators:")
display(df_system_health.limit(10))


In [ ]:
# Forecast system health to predict failures
print("⚠️ Step 6: Forecasting system health for failure prediction...")

# Use ai_forecast to predict system health scores
health_forecast_query = f"""
SELECT * FROM AI_FORECAST(
  TABLE(system_health_ts),
  horizon => '{forecast_horizon}',
  time_col => 'date',
  value_col => ARRAY('health_score', 'total_issues', 'critical_issues'),
  group_col => 'affected_systems',
  prediction_interval_width => 0.85,
  frequency => '1D',
  parameters => '{{"global_floor": 0}}'
)
"""

df_health_forecast = spark.sql(health_forecast_query)

# Identify systems at risk of failure (high health score forecast)
df_failure_risk = df_health_forecast.filter(
    col("health_score_forecast") > 5.0  # Threshold for high risk
).orderBy(desc("health_score_forecast"))

print("✅ System health forecast completed")
print(f"⚠️ {df_failure_risk.count()} high-risk predictions identified")

# Display systems at risk
print("\n⚠️ Systems at risk of failure:")
display(df_failure_risk.select(
    "affected_systems",
    "date",
    "health_score_forecast",
    "health_score_upper",
    "health_score_lower"
).orderBy(desc("health_score_forecast")))


In [ ]:
# Create actionable insights and recommendations
print("💡 Step 7: Generating actionable insights and recommendations...")

# Create the forecast table first
df_health_forecast.createOrReplaceTempView("system_health_forecast")

# Analyze forecast trends and create risk assessment
insights_query = """
WITH forecast_summary AS (
  SELECT 
    affected_systems,
    AVG(health_score_forecast) as avg_predicted_health,
    MAX(health_score_forecast) as max_predicted_health,
    COUNT(*) as forecast_days
  FROM system_health_forecast
  GROUP BY affected_systems
),
risk_assessment AS (
  SELECT 
    affected_systems,
    avg_predicted_health,
    max_predicted_health,
    CASE 
      WHEN max_predicted_health > 8 THEN 'CRITICAL - Immediate attention required'
      WHEN max_predicted_health > 6 THEN 'HIGH - Schedule maintenance soon'
      WHEN max_predicted_health > 4 THEN 'MEDIUM - Monitor closely'
      ELSE 'LOW - Normal operation expected'
    END as risk_level,
    CASE 
      WHEN max_predicted_health > 8 THEN 'Schedule emergency maintenance within 48 hours'
      WHEN max_predicted_health > 6 THEN 'Plan maintenance within 1 week'
      WHEN max_predicted_health > 4 THEN 'Increase monitoring frequency'
      ELSE 'Continue normal monitoring'
    END as recommended_action
  FROM forecast_summary
)
SELECT * FROM risk_assessment
ORDER BY max_predicted_health DESC
"""

df_insights = spark.sql(insights_query)

print("✅ Actionable insights generated")
print(f"💡 Risk assessment for {df_insights.count()} systems")

# Display insights
print("\n💡 System Risk Assessment and Recommendations:")
display(df_insights)


In [ ]:
# Save forecasting results to Unity Catalog
print("💾 Step 8: Saving forecasting results to Unity Catalog...")

# Save overall volume forecast
df_forecast.write.format("delta").mode("overwrite").saveAsTable(
    f"{UNITY_CATALOG['catalog_name']}.{UNITY_CATALOG['schema_name']}.ticket_volume_forecast"
)

# Save system health forecast
df_health_forecast.write.format("delta").mode("overwrite").saveAsTable(
    f"{UNITY_CATALOG['catalog_name']}.{UNITY_CATALOG['schema_name']}.system_health_forecast"
)

# Save risk assessment insights
df_insights.write.format("delta").mode("overwrite").saveAsTable(
    f"{UNITY_CATALOG['catalog_name']}.{UNITY_CATALOG['schema_name']}.risk_assessment_insights"
)

print("✅ All forecasting results saved to Unity Catalog")
print("\n📊 Saved tables:")
print(f"   • {UNITY_CATALOG['catalog_name']}.{UNITY_CATALOG['schema_name']}.ticket_volume_forecast")
print(f"   • {UNITY_CATALOG['catalog_name']}.{UNITY_CATALOG['schema_name']}.system_health_forecast")
print(f"   • {UNITY_CATALOG['catalog_name']}.{UNITY_CATALOG['schema_name']}.risk_assessment_insights")

print("\n🎯 Ready for Streamlit dashboard integration!")


In [ ]:
# Summary of forecasting capabilities
print("🔮 AI FORECASTING SUMMARY")
print("=" * 50)
print("\n✅ Successfully implemented Databricks ai_forecast function")
print("\n📈 Forecasting Capabilities:")
print("   • Ticket volume prediction (30-day horizon)")
print("   • System health score prediction")
print("   • Asset failure risk assessment")
print("   • Proactive maintenance recommendations")

print("\n🎯 Key Features:")
print("   • Prophet-like piecewise linear modeling")
print("   • Automatic seasonality detection")
print("   • 95% prediction intervals")
print("   • Multi-metric and multi-group forecasting")
print("   • Built-in confidence intervals")

print("\n💡 Business Value:")
print("   • Proactive issue prevention")
print("   • Resource planning optimization")
print("   • Reduced downtime and costs")
print("   • Data-driven maintenance scheduling")
print("   • Improved system reliability")

print("\n🚀 Next Steps:")
print("   • Integrate forecasts into Streamlit dashboard")
print("   • Set up automated forecasting jobs")
print("   • Create alerting for high-risk predictions")
print("   • Implement real-time model retraining")

print("\n🎉 AI Forecasting implementation complete!")
